# 📊 EXPLORATORY DATA ANALYSIS - HOUSE PRICE PREDICTION
## Mục tiêu: Phân tích trực quan dữ liệu, phát hiện missing values, outliers

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Thiết lập style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Đường dẫn tự động xác định (hỗ trợ CWD từ prj/eda, ml_hybrid_project, hoặc workspace root)
possible_raw_dirs = [
    os.path.abspath('../../data/raw'),           # Khi CWD là prj/eda/
    os.path.abspath('../data/raw'),              # Khi CWD là prj/
    os.path.abspath('data/raw'),                 # Khi CWD là ml_hybrid_project/
    os.path.abspath('lab03-competition/ml_hybrid_project/data/raw') # Khi CWD là workspace root
]

BASE_DIR = None
for path in possible_raw_dirs:
    if os.path.exists(os.path.join(path, 'train.csv')):
        BASE_DIR = path
        break

if BASE_DIR is None:
    raise FileNotFoundError("Không tìm thấy thư mục data/raw chứa train.csv!")

PROJECT_ROOT = os.path.dirname(os.path.dirname(BASE_DIR))
MODEL_DIR = os.path.join(PROJECT_ROOT, 'prj', 'model')
os.makedirs(MODEL_DIR, exist_ok=True)

train_path = os.path.join(BASE_DIR, 'train.csv')
test_path = os.path.join(BASE_DIR, 'test.csv')

# Đọc dữ liệu
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print('=' * 60)
print('TẬP DỮ LIỆU')
print('=' * 60)
print(f'Train shape: {train_df.shape}')
print(f'Test shape:  {test_df.shape}')
print(f'Data dir:    {BASE_DIR}')
print(f'Model dir:   {MODEL_DIR}')

---
## 1. PHÂN TÍCH BIẾN MỤC TIÊU (SalePrice)

In [ ]:
# Biến mục tiêu
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1.1. Phân phối SalePrice
sns.histplot(train_df['SalePrice'], kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Phân phối SalePrice', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Giá nhà ($)')
axes[0].set_ylabel('Tần suất')

# 1.2. Boxplot SalePrice
sns.boxplot(y=train_df['SalePrice'], ax=axes[1], color='coral')
axes[1].set_title('Boxplot SalePrice - Phát hiện Outliers', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Giá nhà ($)')

# 1.3. Log-transformed SalePrice
sale_log = np.log1p(train_df['SalePrice'])
sns.histplot(sale_log, kde=True, ax=axes[2], color='seagreen')
axes[2].set_title('Phân phối log(SalePrice) - Bình thường hóa', fontsize=14, fontweight='bold')
axes[2].set_xlabel('log(Giá nhà)')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'saleprice_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 THỐNG KÊ SALEPRICE:')
print(train_df['SalePrice'].describe())

---
## 2. PHÂN TÍCH MISSING VALUES

In [ ]:
# Missing values trong train
missing = train_df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(train_df) * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 2.1. Biểu đồ cột ngang - Missing Values
colors = ['#ff6b6b' if v > 50 else '#ffd93d' if v > 20 else '#6bcb77' 
          for v in missing_pct.values]
bars = axes[0].barh(missing.index, missing.values, color=colors)
axes[0].set_xlabel('Số lượng Missing Values')
axes[0].set_title('Missing Values trong Train (Before Preprocessing)', 
                  fontsize=14, fontweight='bold')
for bar, val, pct in zip(bars, missing.values, missing_pct.values):
    axes[0].text(val + 5, bar.get_y() + bar.get_height()/2,
                 f'{val} ({pct}%)', va='center', fontsize=9)

# 2.2. Heatmap missing values (top 10)
top_missing = missing.head(10)
missing_df = train_df[top_missing.index].isnull()
sns.heatmap(missing_df, cmap='RdYlGn_r', cbar=False, ax=axes[1],
            yticklabels=False)
axes[1].set_title('Heatmap Missing Values (Top 10 Columns)', 
                  fontsize=14, fontweight='bold')
axes[1].set_xlabel('Features')
axes[1].set_ylabel('Samples')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'missing_values_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\n📋 BẢNG MISSING VALUES:')
print(pd.DataFrame({'Missing': missing, 'Percent': missing_pct}).head(15))

---
## 3. PHÂN TÍCH OUTLIERS

In [ ]:
# Outliers: GrLivArea vs SalePrice
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 3.1. Scatter: GrLivArea vs SalePrice
axes[0].scatter(train_df['GrLivArea'], train_df['SalePrice'], 
               alpha=0.5, c='steelblue')
axes[0].set_xlabel('Above Grade Living Area (sqft)')
axes[0].set_ylabel('SalePrice ($)')
axes[0].set_title('GrLivArea vs SalePrice - Phát hiện Outliers', 
                  fontsize=14, fontweight='bold')

# Đánh dấu outliers
outliers = train_df[(train_df['GrLivArea'] > 4000) & (train_df['SalePrice'] < 300000)]
axes[0].scatter(outliers['GrLivArea'], outliers['SalePrice'], 
               color='red', s=100, label='Outliers', zorder=5)
axes[0].legend()

# 3.2. Scatter: OverallQual vs SalePrice
axes[1].scatter(train_df['OverallQual'], train_df['SalePrice'], 
               alpha=0.5, c='coral')
axes[1].set_xlabel('Overall Quality (1-10)')
axes[1].set_ylabel('SalePrice ($)')
axes[1].set_title('Overall Quality vs SalePrice', 
                  fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'outliers_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'\n⚠️ Số outliers phát hiện (GrLivArea > 4000 & SalePrice < 300k): {len(outliers)}')

---
## 4. CORRELATION ANALYSIS

In [ ]:
# Ma trận tương quan với SalePrice
numeric_cols = train_df.select_dtypes(include=[np.number]).columns
corr_with_target = train_df[numeric_cols].corr()['SalePrice'].drop('SalePrice').sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# 4.1. Top 15 features tương quan cao nhất
top_corr = corr_with_target.head(15)
colors = ['#ff6b6b' if v < 0 else '#6bcb77' for v in top_corr.values]
bars = axes[0].barh(top_corr.index[::-1], top_corr.values[::-1], color=colors[::-1])
axes[0].axvline(x=0, color='black', linewidth=0.8)
axes[0].set_xlabel('Correlation with SalePrice')
axes[0].set_title('Top 15 Features tương quan cao nhất với SalePrice', 
                  fontsize=14, fontweight='bold')

# 4.2. Heatmap correlation matrix (top features)
top_features = corr_with_target.head(10).index.tolist() + ['SalePrice']
corr_matrix = train_df[top_features].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=axes[1], linewidths=0.5,
            cbar_kws={'label': 'Correlation'})
axes[1].set_title('Correlation Heatmap (Top Features)', 
                  fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'correlation_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\n🔝 TOP 10 FEATURES TƯƠNG QUAN VỚI SALEPRICE:')
print(corr_with_target.head(10))

---
## 5. PHÂN TÍCH THEO NHÓM (Neighborhood & Quality)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 5.1. Giá nhà theo Neighborhood
neighborhood_price = train_df.groupby('Neighborhood')['SalePrice'].median().sort_values()
axes[0,0].barh(neighborhood_price.index, neighborhood_price.values, color='steelblue')
axes[0,0].set_xlabel('Median SalePrice ($)')
axes[0,0].set_title('Median Giá nhà theo Khu vực (Neighborhood)', 
                    fontsize=12, fontweight='bold')
axes[0,0].tick_params(axis='y', labelsize=8)

# 5.2. Giá nhà theo OverallQual
quality_price = train_df.groupby('OverallQual')['SalePrice'].median()
axes[0,1].bar(quality_price.index, quality_price.values, color='coral')
axes[0,1].set_xlabel('Overall Quality')
axes[0,1].set_ylabel('Median SalePrice ($)')
axes[0,1].set_title('Median Giá nhà theo Chất lượng tổng thể', 
                    fontsize=12, fontweight='bold')

# 5.3. Giá nhà theo YearBuilt
year_price = train_df.groupby('YearBuilt')['SalePrice'].median()
axes[1,0].scatter(year_price.index, year_price.values, alpha=0.6, color='seagreen')
axes[1,0].set_xlabel('Year Built')
axes[1,0].set_ylabel('Median SalePrice ($)')
axes[1,0].set_title('Median Giá nhà theo Năm xây dựng', 
                    fontsize=12, fontweight='bold')

# 5.4. Giá nhà theo YearRemodAdd
remod_price = train_df.groupby('YearRemodAdd')['SalePrice'].median()
axes[1,1].scatter(remod_price.index, remod_price.values, alpha=0.6, color='purple')
axes[1,1].set_xlabel('Year Remodeled')
axes[1,1].set_ylabel('Median SalePrice ($)')
axes[1,1].set_title('Median Giá nhà theo Năm cải tạo', 
                    fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'neighborhood_quality_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 6. FEATURE DISTRIBUTIONS

In [ ]:
# Phân phối các features quan trọng
key_features = ['GrLivArea', 'TotalBsmtSF', 'GarageArea', '1stFlrSF', 'TotRmsAbvGrd']
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    if feat in train_df.columns:
        sns.histplot(train_df[feat].dropna(), kde=True, ax=axes[i], color='steelblue')
        axes[i].set_title(f'Phân phối {feat}', fontsize=11, fontweight='bold')
        axes[i].set_xlabel(feat)

# Ẩn axes thừa
for j in range(len(key_features), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'feature_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## ✅ KẾT LUẬN EDA

In [ ]:
print('=' * 60)
print('KẾT LUẬN PHÂN TÍCH DỮ LIỆU')
print('=' * 60)
print('''
📊 TỔNG QUAN:
   • Train: 1460 samples × 81 features
   • Test: 1459 samples × 80 features

⚠️ MISSING VALUES:
   • PoolQC, MiscFeature, Alley, Fence: >80% missing → NA = "None"
   • Garage*: ~5% missing → NA = "None" / 0
   • LotFrontage: ~17% missing → median imputation

🔝 TOP FEATURES TƯƠNG QUAN VỚI SALEPRICE:
   1. OverallQual (0.79) - Chất lượng tổng thể
   2. GrLivArea (0.71) - Diện tích sống
   3. GarageCars (0.64) - Số xe trong garage
   4. GarageArea (0.62) - Diện tích garage
   5. TotalBsmtSF (0.61) - Tổng diện tích basement

📐 OUTLIERS:
   • GrLivArea > 4000 sqft + SalePrice < 300k → Cần xử lý

📈 TARGET DISTRIBUTION:
   • SalePrice bị lệch phải (right-skewed)
   • Nên dùng log-transform cho training
''')

print(f'✅ Đã lưu các biểu đồ vào thư mục: {MODEL_DIR}')
